# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata.get('name')}: {metadata.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets and their fields:")
record_set_dicts = []
for rs in dataset.record_sets:
    record_set_id = rs['@id']
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = [fld['@id'] if isinstance(fld, dict) and '@id' in fld else str(fld) for fld in fields]
    print(f"Record set @id: {record_set_id}")
    print(f"  Fields: {field_ids}")
    record_set_dicts.append({'@id': record_set_id, 'fields': field_ids})

# Example: Print a small sample record from each record set
for rs in dataset.record_sets:
    recs = dataset.records(record_set=rs['@id'])
    try:
        print(f"\nSample record for record set {rs['@id']}:")
        print(next(recs))
    except StopIteration:
        print(f"No records found for record set {rs['@id']}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

<sup>Note: In the FAIR² dataset, there is generally a main record set containing the clinical data table. We will identify and extract that record set below by its `@id`.</sup>

In [ ]:
# Identify all record sets by @id
record_sets = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set @ids found: {record_sets}")

# Load each record set into a DataFrame
dataframes = dict()
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Print columns of the main clinical data record set (guessing which one it is, usually has more columns)
main_rs_id = None
max_cols = 0
for rsid, df in dataframes.items():
    print(f"Record set {rsid}: {df.shape[1]} columns, {df.shape[0]} rows")
    if df.shape[1] > max_cols:
        main_rs_id = rsid
        max_cols = df.shape[1]

print(f"\nColumns in main record set ({main_rs_id}): \n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We will select relevant fields for analysis (referenced by their `@id` from the columns above), filter, normalize, and group the data. Typical operations include removing outliers, transforming values, and summarizing by categorical attributes.

**Note**: All variable references use the field `@id` as shown in the dataset.

In [ ]:
# Choose a numeric field and group field by their full @id
# List all available columns first
cols = dataframes[main_rs_id].columns.tolist()
print("Columns in main DataFrame:", cols)

# Typical numeric fields may represent age, intervals, etc. Let's try to select one that contains 'age' or 'interval'
numeric_field = None
group_field = None
for col in cols:
    if 'age' in col.lower():
        numeric_field = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field = col

# Fallbacks
if numeric_field is None:
    numeric_field = cols[0]  # Just use the first
if group_field is None:
    for col in cols:
        if 'anatomical' in col.lower() or 'location' in col.lower():
            group_field = col
            break

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Remove obviously invalid values (e.g., age <= 0)
threshold = 10
df = dataframes[main_rs_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by the group_field if possible
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll show the distribution of the selected numeric field and compare by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field], bins=15, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group if group_field exists
if group_field in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:  
- Loaded and inspected the FAIR² colorectal cancer survivors dataset using `mlcroissant` and the Croissant schema.  
- Explored its record sets, referencing all fields and entities using their `@id` identifiers for precise programmatic access.  
- Performed exploratory analysis, filtering, normalization, grouping, and visualizations using variable fields by their `@id`.  

Further analysis may involve deeper clinical subsetting or machine learning applications based on these processed data.